In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from datetime import datetime

bitcoin_df_daily = pd.read_csv("BTC_history_daily.csv")
bitcoin_df_daily['time'] = pd.to_datetime(bitcoin_df_daily['time'])

# Select data for a specific year (e.g., 2024)
bitcoin2024 = bitcoin_df_daily[bitcoin_df_daily['time'].dt.year == 2024]

# Create lagged features and technical indicators
def add_features(df):
    for lag in range(1, 6):
        df[f'close_lag_{lag}'] = df['close'].shift(lag)
    df.dropna(inplace=True)
    return df

columns_to_drop = ['conversionType', 'conversionSymbol']
bitcoin2024 = bitcoin2024.drop(columns=columns_to_drop)
bitcoin_df_daily_f = add_features(bitcoin2024)
bitcoin_df_daily_f = bitcoin_df_daily_f.set_index('time')

# --- ตรวจสอบฤดูกาลใน Time Series ---
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf

bitcoin_close = bitcoin_df_daily_f['close']

# Seasonal Decomposition
seasonal_result = seasonal_decompose(bitcoin_close, model='additive', period=30)
seasonal_result.plot()
plt.suptitle('Seasonal Decomposition of Bitcoin Prices (2024)', fontsize=16)
plt.tight_layout()
plt.show()

# Autocorrelation Plot
plot_acf(bitcoin_close, lags=60)
plt.title('Autocorrelation of Bitcoin Prices (2024)')
plt.show()

# Modeling
features = ['close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_4', 'close_lag_5']
target = 'close'
test_size = 0.5
train, test = train_test_split(bitcoin_df_daily_f['close'], test_size=test_size, shuffle=False)

# SARIMAX
sarimax_model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 60))
sarimax_fit = sarimax_model.fit()
sarimax_forecast = sarimax_fit.forecast(len(test))

# Linear Regression
train_lr, test_lr = train_test_split(bitcoin_df_daily_f, test_size=test_size, shuffle=False)
X_train_lr = train_lr[features]
y_train_lr = train_lr[target]
X_test_lr = test_lr[features]
y_test_lr = test_lr[target]

lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)
lr_forecast = lr_model.predict(X_test_lr)

# Holt-Winters
holt_model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=90)
holt_fit = holt_model.fit()
holt_forecast = holt_fit.forecast(len(test))

# Evaluation
from sklearn.metrics import mean_absolute_percentage_error as mape

results = pd.DataFrame({
    'Model': ['SARIMAX', 'Linear Regression', 'Holt-Winters'],
    'RMSE': [np.sqrt(mean_squared_error(test, sarimax_forecast)),
             np.sqrt(mean_squared_error(y_test_lr, lr_forecast)),
             np.sqrt(mean_squared_error(test, holt_forecast))],
    'MAE': [mean_absolute_error(test, sarimax_forecast),
            mean_absolute_error(y_test_lr, lr_forecast),
            mean_absolute_error(test, holt_forecast)],
    'MAPE': [mape(test, sarimax_forecast)*100,
             mape(y_test_lr, lr_forecast)*100,
             mape(test, holt_forecast)*100]
})
print(results)

# Plot Forecasts
plt.figure(figsize=(14, 7))
plt.plot(test_lr.index, test_lr['close'], label='Actual')
plt.plot(test_lr.index[1:], sarimax_forecast[1:], label='SARIMAX')
plt.plot(test_lr.index, lr_forecast, label='Linear Regression')
plt.plot(test_lr.index[1:], holt_forecast[1:], label='Holt-Winters')
plt.title('Forecast Comparison')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
